In [ ]:
"""
Knowledge Distillation — Script 5: Online Distillation
=======================================================
Models  : ResNet-50 × N peers (multi-student, same modified CIFAR-10 arch)
Dataset : CIFAR-10
Method  : Online Distillation — all models train simultaneously, no pre-trained teacher

WHAT IS ONLINE DISTILLATION?
==============================
No pre-trained teacher needed. A cohort of N models trains in parallel, each
simultaneously acting as a student (learning from peers) and a teacher
(providing soft targets to peers). The "teacher" emerges dynamically from
the evolving ensemble.

Two variants:

1. DML — Deep Mutual Learning (Zhang 2018)
   Each model i minimises:
     L_i = CE(y, p_i) + (1/(N-1)) * Σ_{j≠i} KL(p_j || p_i)
   All peer KL terms use detached probabilities from the other models,
   so only model i's graph is active during model i's backward pass.

2. OKDDip — Online KD with Diverse Peers (Chen 2020)
   Adds a group-level teacher (soft ensemble average):
     p_group = (1/N) * Σ_i softmax(logits_i / tau)
     L_i = CE(y, p_i) + KL(p_peers || p_i) + KL(p_group || p_i)
   The ensemble is a consistently stronger teacher than any single peer.

KEY DESIGN — per-peer GradScaler:
  Each peer has its own GradScaler AND optimizer. Sharing one scaler across
  N backward passes is incorrect: the scale gets updated N times per batch,
  causing the loss scale to collapse near zero after a few hundred batches.
  With per-peer scalers each backward pass is fully self-contained and the
  graphs from OTHER peers are always .detach()-ed, so retain_graph=False.

VRAM:
  Each ResNet-50 (batch=128, AMP) uses ~5.5 GB peak on Blackwell.
  The script auto-computes the VRAM-safe peer cap and warns/skips if exceeded.
    N=2 → ~11 GB   N=3 → ~16.5 GB   N=4 → ~22 GB

Epochs: 15  (online KD starts from random init — needs more than offline KD)

Sweeps:
  A — variant     : [dml, okddip]           (N=min(3,max_safe), tau=3)
  B — N peers     : [2, 3, 4]*              (best variant, tau=3)
  C — temperature : [2, 3, 4]               (best variant + best N)
  * auto-capped by VRAM

Output : __5_v2_KD_online_distillation.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch  {torch.__version__}", flush=True)
print(f"[init] Device   : {DEVICE}", flush=True)
if DEVICE.type == "cuda":
    _props = torch.cuda.get_device_properties(0)
    TOTAL_VRAM_GB = _props.total_memory / 1024**3
    print(f"[init] GPU      : {torch.cuda.get_device_name(0)}", flush=True)
    print(f"[init] VRAM     : {TOTAL_VRAM_GB:.1f} GB", flush=True)
else:
    TOTAL_VRAM_GB = 0.0

# ── VRAM guard ─────────────────────────────────────────────────────────────────
_VRAM_PER_PEER_GB = 5.5   # peak per ResNet-50, batch=128, AMP
_VRAM_HEADROOM_GB = 1.5   # reserve for PyTorch runtime

def _max_safe_peers() -> int:
    if DEVICE.type != "cuda":
        return 2
    return max(2, int((TOTAL_VRAM_GB - _VRAM_HEADROOM_GB) / _VRAM_PER_PEER_GB))

MAX_PEERS = _max_safe_peers()
print(f"[init] Max safe peers : {MAX_PEERS}", flush=True)

# ── Config ─────────────────────────────────────────────────────────────────────
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__5_v2_KD_online_distillation.json"

BATCH_SIZE  = 128 if DEVICE.type == "cuda" else 32
NUM_WORKERS = 0
PIN_MEMORY  = False
NUM_CLASSES = 10
KD_EPOCHS   = 10      # starts from random init → needs more epochs
USE_AMP     = (DEVICE.type == "cuda")

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] Epochs  : {KD_EPOCHS}  AMP: {USE_AMP}", flush=True)


# ── Model ──────────────────────────────────────────────────────────────────────
def build_resnet50_cifar(num_classes: int = 10) -> nn.Module:
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model


# ── Data ───────────────────────────────────────────────────────────────────────
def get_train_loader() -> DataLoader:
    t = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=True, download=True, transform=t)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

def get_test_loader() -> DataLoader:
    t = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False, download=True, transform=t)
    return DataLoader(ds, batch_size=512, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


# ── Helpers ────────────────────────────────────────────────────────────────────
def model_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)

def param_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    yp, yt = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(yt, yp)),
        "precision": float(precision_score(yt, yp, average="macro", zero_division=0)),
        "recall"   : float(recall_score(yt, yp, average="macro", zero_division=0)),
        "f1"       : float(f1_score(yt, yp, average="macro", zero_division=0)),
    }

def measure_inference_ms(model: nn.Module, runs: int = 30) -> float:
    m = copy.deepcopy(model).to(DEVICE).eval()
    dummy = torch.randn(128, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        for _ in range(5): m(dummy)
        sync()
        t0 = time.perf_counter()
        for _ in range(runs): m(dummy)
        sync()
    return (time.perf_counter() - t0) / runs * 1000

def evaluate_ensemble(peers: list, loader: DataLoader) -> float:
    for p in peers: p.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, lbls in loader:
            inputs = inputs.to(DEVICE, non_blocking=True)
            probs = torch.stack([
                F.softmax(p(inputs), dim=1).cpu() for p in peers
            ]).mean(0)
            all_preds.extend(probs.argmax(1).numpy())
            all_labels.extend(lbls.numpy())
    return float(accuracy_score(np.array(all_labels), np.array(all_preds)))


# ══════════════════════════════════════════════════════════════════════════════
# Core training loop
# ══════════════════════════════════════════════════════════════════════════════
def run_online_kd(
    train_loader : DataLoader,
    test_loader  : DataLoader,
    baseline_acc : float,
    variant      : str   = "dml",
    n_peers      : int   = 3,
    tau          : float = 3.0,
    lr           : float = 0.1,
    n_epochs     : int   = KD_EPOCHS,
    sweep_name   : str   = "variant",
) -> dict:

    # Clamp to VRAM-safe limit
    if n_peers > MAX_PEERS:
        print(f"  [WARN] N={n_peers} → capped to {MAX_PEERS} (VRAM limit)", flush=True)
        n_peers = MAX_PEERS

    tag = f"{variant}/N={n_peers}/tau={tau}"
    print(f"\n  ┌─ [{tag}]", flush=True)

    try:
        # ── Build peers ───────────────────────────────────────────────────────
        peers = [build_resnet50_cifar(NUM_CLASSES).to(DEVICE) for _ in range(n_peers)]
        print(f"      [init] {n_peers} × ResNet-50  (~{n_peers*_VRAM_PER_PEER_GB:.0f} GB VRAM)",
              flush=True)

        # Per-peer optimizers, schedulers, and — critically — per-peer scalers
        optimizers = [
            torch.optim.SGD(p.parameters(), lr=lr, momentum=0.9,
                            weight_decay=1e-4, nesterov=True)
            for p in peers
        ]
        schedulers = [
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
            for opt in optimizers
        ]
        scalers = [torch.amp.GradScaler(enabled=USE_AMP) for _ in peers]

        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        peer_epoch_acc = [[] for _ in range(n_peers)]
        t_start = time.perf_counter()

        for epoch in range(n_epochs):
            for p in peers: p.train()
            correct_per_peer = [0] * n_peers
            total = 0
            t_ep  = time.perf_counter()

            for batch_idx, (inputs, targets) in enumerate(train_loader):
                inputs  = inputs.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)

                # ── Forward all peers to get logits ───────────────────────
                # We forward inside autocast but keep each peer's logit tensor
                # alive (attached graph) for its own backward.
                # Other peers' logits are immediately detached for KL targets.
                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    all_logits = [p(inputs) for p in peers]  # (B,C) each

                    # Detached soft probs — used as KL targets for ALL peers
                    all_probs_det = [
                        F.softmax(lg.detach() / tau, dim=1) for lg in all_logits
                    ]

                    # OKDDip group teacher (fully detached)
                    group_prob = (torch.stack(all_probs_det).mean(0)
                                  if variant == "okddip" else None)

                # ── Per-peer backward (no retain_graph needed) ────────────
                for i in range(n_peers):
                    optimizers[i].zero_grad(set_to_none=True)

                    with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                        ce_i = F.cross_entropy(all_logits[i], targets)

                        # Sum KL from every other peer (all detached)
                        kl_sum = torch.zeros(1, device=DEVICE)
                        for j in range(n_peers):
                            if j == i:
                                continue
                            log_pi = F.log_softmax(all_logits[i] / tau, dim=1)
                            kl_sum = kl_sum + F.kl_div(
                                log_pi, all_probs_det[j],
                                reduction="batchmean"
                            ) * (tau ** 2)

                        kl_avg = kl_sum / max(n_peers - 1, 1)

                        if variant == "okddip":
                            log_pi  = F.log_softmax(all_logits[i] / tau, dim=1)
                            kl_grp  = F.kl_div(log_pi, group_prob,
                                               reduction="batchmean") * (tau ** 2)
                            loss_i  = ce_i + kl_avg + kl_grp
                        else:
                            loss_i = ce_i + kl_avg

                    scalers[i].scale(loss_i).backward()
                    scalers[i].step(optimizers[i])
                    scalers[i].update()

                with torch.no_grad():
                    for i in range(n_peers):
                        correct_per_peer[i] += (
                            all_logits[i].detach().argmax(1) == targets
                        ).sum().item()
                total += targets.size(0)

                if (batch_idx + 1) % 100 == 0:
                    elapsed = time.perf_counter() - t_ep
                    done    = batch_idx + 1
                    eta     = elapsed / done * (len(train_loader) - done)
                    accs    = " | ".join(
                        f"p{i}={correct_per_peer[i]/total:.3f}" for i in range(n_peers)
                    )
                    print(f"      [epoch {epoch+1}/{n_epochs}] "
                          f"batch {done}/{len(train_loader)}  {accs}  ETA={eta:.0f}s",
                          flush=True)

            for sch in schedulers: sch.step()
            sync()

            ep_time   = time.perf_counter() - t_ep
            remaining = n_epochs - (epoch + 1)
            acc_vals  = [round(correct_per_peer[i] / total, 6) for i in range(n_peers)]
            for i in range(n_peers):
                peer_epoch_acc[i].append(acc_vals[i])

            print(f"      [epoch {epoch+1}/{n_epochs}] DONE  "
                  f"peer_accs={acc_vals}  "
                  f"time={ep_time:.1f}s  ETA_remaining={ep_time*remaining:.0f}s",
                  flush=True)

        sync()
        train_time_s = time.perf_counter() - t_start
        peak_gpu_mb  = (round(torch.cuda.max_memory_allocated() / 1024**2, 2)
                        if DEVICE.type == "cuda" else None)

        # ── Evaluate ──────────────────────────────────────────────────────────
        print("      [eval] Evaluating all peers ...", flush=True)
        peer_metrics = []
        for i, peer in enumerate(peers):
            m = evaluate(peer, test_loader)
            peer_metrics.append(m)
            print(f"             Peer {i}: acc={m['accuracy']:.4f}", flush=True)

        best_idx     = max(range(n_peers), key=lambda i: peer_metrics[i]["accuracy"])
        best_metrics = peer_metrics[best_idx]
        avg_acc      = float(np.mean([m["accuracy"] for m in peer_metrics]))

        print("      [eval] Ensemble ...", flush=True)
        ens_acc = evaluate_ensemble(peers, test_loader)
        print(f"             Ensemble: acc={ens_acc:.4f}", flush=True)

        inference_ms = measure_inference_ms(peers[best_idx])
        size_mb      = model_size_mb(peers[best_idx])
        acc_drop     = baseline_acc - best_metrics["accuracy"]
        ens_drop     = baseline_acc - ens_acc

        print(f"  └─ BestPeer={best_metrics['accuracy']:.4f}  "
              f"Ensemble={ens_acc:.4f}  Drop={acc_drop:+.4f}  "
              f"Train={train_time_s:.1f}s", flush=True)

        return {
            "sweep"             : sweep_name,
            "variant"           : variant,
            "n_peers"           : n_peers,
            "temperature"       : tau,
            "lr"                : lr,
            "lr_schedule"       : "cosine",
            "epochs"            : n_epochs,
            "train_device"      : str(DEVICE),
            "use_amp"           : USE_AMP,
            "train_time_s"      : round(train_time_s, 2),
            "peer_epoch_acc"    : peer_epoch_acc,
            "peer_final_acc"    : [m["accuracy"] for m in peer_metrics],
            "best_peer_idx"     : best_idx,
            "accuracy"          : round(best_metrics["accuracy"],  6),
            "ensemble_accuracy" : round(ens_acc, 6),
            "avg_peer_accuracy" : round(avg_acc, 6),
            "precision"         : round(best_metrics["precision"], 6),
            "recall"            : round(best_metrics["recall"],    6),
            "f1"                : round(best_metrics["f1"],        6),
            "accuracy_drop"     : round(acc_drop, 6),
            "ensemble_drop"     : round(ens_drop, 6),
            "size_mb"           : round(size_mb, 4),
            "inference_ms"      : round(inference_ms, 4),
            "peak_gpu_memory_mb": peak_gpu_mb,
            "params_per_model"  : param_count(peers[0]),
            "description"       : (
                f"Online KD ({variant}, N={n_peers}, tau={tau}, "
                f"cosine, epochs={n_epochs})"
            ),
            "status": "ok",
        }

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return {
            "sweep": sweep_name, "variant": variant,
            "n_peers": n_peers, "temperature": tau,
            "status": "failed", "error": str(e),
            "accuracy": None, "accuracy_drop": None,
        }


# ── Main ───────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Online Distillation  (DML + OKDDip)")
    print(f"  Models : ResNet-50 × N peers — no pre-trained teacher")
    print(f"  Device : {DEVICE}  |  Epochs : {KD_EPOCHS}  |  Batch : {BATCH_SIZE}")
    print(f"  AMP    : {USE_AMP}  |  Max safe peers : {MAX_PEERS}")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline acc : {baseline_acc:.4f}\n", flush=True)

    ref = build_resnet50_cifar(NUM_CLASSES)
    ref_size_mb = model_size_mb(ref)
    ref_inf_ms  = measure_inference_ms(ref)
    print(f"  Per-peer — Size: {ref_size_mb:.2f} MB  "
          f"Params: {param_count(ref):,}  Inf: {ref_inf_ms:.1f} ms\n", flush=True)

    train_loader = get_train_loader()
    test_loader  = get_test_loader()

    results = []
    n_sweep_a = min(3, MAX_PEERS)

    # ── Sweep A: Variant ──────────────────────────────────────────────────────
    print(f"  ── Sweep A : Variant  (N={n_sweep_a}, tau=3) ─────────────────────",
          flush=True)
    sweep_a = []
    for variant in ["dml", "okddip"]:
        row = run_online_kd(
            train_loader, test_loader, baseline_acc,
            variant=variant, n_peers=n_sweep_a, tau=3.0,
            lr=0.1, n_epochs=KD_EPOCHS, sweep_name="A_variant",
        )
        results.append(row)
        sweep_a.append(row)

    valid_a = [r for r in sweep_a if r.get("accuracy") is not None]
    if not valid_a:
        print("  ✗ Sweep A failed entirely.", flush=True)
        return
    best_variant = max(valid_a, key=lambda r: r["accuracy"])["variant"]
    print(f"\n  Best variant : {best_variant}", flush=True)

    # ── Sweep B: N peers ──────────────────────────────────────────────────────
    print(f"\n  ── Sweep B : N peers  (variant={best_variant}, tau=3) ────────────",
          flush=True)
    sweep_b = []
    for n_peers in [2, 3, 4]:
        if n_peers > MAX_PEERS:
            print(f"  [SKIP] N={n_peers} exceeds safe limit ({MAX_PEERS})", flush=True)
            continue
        row = run_online_kd(
            train_loader, test_loader, baseline_acc,
            variant=best_variant, n_peers=n_peers, tau=3.0,
            lr=0.1, n_epochs=KD_EPOCHS, sweep_name="B_n_peers",
        )
        results.append(row)
        sweep_b.append(row)

    valid_b = [r for r in sweep_b if r.get("accuracy") is not None]
    best_n  = (max(valid_b, key=lambda r: r["accuracy"])["n_peers"]
               if valid_b else n_sweep_a)
    print(f"\n  Best N : {best_n}", flush=True)

    # ── Sweep C: Temperature ──────────────────────────────────────────────────
    print(f"\n  ── Sweep C : Temperature  (variant={best_variant}, N={best_n}) ───",
          flush=True)
    for tau in [2.0, 3.0, 4.0]:
        row = run_online_kd(
            train_loader, test_loader, baseline_acc,
            variant=best_variant, n_peers=best_n, tau=tau,
            lr=0.1, n_epochs=KD_EPOCHS, sweep_name="C_temperature",
        )
        results.append(row)

    # ── Best overall ──────────────────────────────────────────────────────────
    valid = [r for r in results if r.get("accuracy") is not None]
    if not valid:
        print("  ✗ No valid results.", flush=True)
        return
    best = max(valid, key=lambda r: r["accuracy"])

    report = {
        "method"        : "online_distillation",
        "description"   : (
            "Online Distillation — DML / OKDDip "
            "(N ResNet-50 peers co-trained, no pre-trained teacher)"
        ),
        "model_arch"    : "ResNet-50 × N (CIFAR-10 modified, 3×3 conv1, no maxpool)",
        "dataset"       : "CIFAR-10",
        "train_device"  : str(DEVICE),
        "use_amp"       : USE_AMP,
        "kd_epochs"     : KD_EPOCHS,
        "max_safe_peers": MAX_PEERS,
        "baseline"      : baseline,
        "model_info"    : {
            "size_mb"     : round(ref_size_mb, 4),
            "inference_ms": round(ref_inf_ms, 4),
            "params"      : param_count(ref),
        },
        "best_config"   : {
            "variant"          : best["variant"],
            "n_peers"          : best["n_peers"],
            "temperature"      : best["temperature"],
            "accuracy"         : best["accuracy"],
            "ensemble_accuracy": best.get("ensemble_accuracy"),
            "accuracy_drop"    : best["accuracy_drop"],
            "size_mb"          : best["size_mb"],
            "inference_ms"     : best["inference_ms"],
        },
        "sweeps"        : {
            "A_variant"    : f"dml vs okddip  (N={n_sweep_a}, tau=3)",
            "B_n_peers"    : "N in [2,3,4] capped by VRAM  (best variant, tau=3)",
            "C_temperature": "tau in [2,3,4]  (best variant + best N)",
        },
        "results"       : results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  ✓ Saved → {OUTPUT_JSON}")
    print(f"  Best : variant={best['variant']} | N={best['n_peers']} | "
          f"tau={best['temperature']} | Acc={best['accuracy']:.4f} | "
          f"Drop={best['accuracy_drop']:+.4f} | "
          f"Ensemble={best.get('ensemble_accuracy', 0.0):.4f}")


if __name__ == "__main__":
    main()

[init] PyTorch  2.12.0.dev20260324+cu128
[init] Device   : cuda
[init] GPU      : NVIDIA GeForce RTX 5070 Laptop GPU
[init] VRAM     : 8.0 GB
[init] Max safe peers : 2
[init] Epochs  : 15  AMP: True

  Online Distillation  (DML + OKDDip)
  Models : ResNet-50 × N peers — no pre-trained teacher
  Device : cuda  |  Epochs : 15  |  Batch : 128
  AMP    : True  |  Max safe peers : 2

  Baseline acc : 0.9320

  Per-peer — Size: 90.03 MB  Params: 23,520,842  Inf: 49.0 ms

  ── Sweep A : Variant  (N=2, tau=3) ─────────────────────

  ┌─ [dml/N=2/tau=3.0]
      [init] 2 × ResNet-50  (~11 GB VRAM)
      [epoch 1/15] batch 100/391  p0=0.099 | p1=0.098  ETA=76s
      [epoch 1/15] batch 200/391  p0=0.101 | p1=0.100  ETA=54s
      [epoch 1/15] batch 300/391  p0=0.101 | p1=0.100  ETA=27s
      [epoch 1/15] DONE  peer_accs=[0.10008, 0.0995]  time=115.8s  ETA_remaining=1621s
      [epoch 2/15] batch 100/391  p0=0.098 | p1=0.098  ETA=89s
      [epoch 2/15] batch 200/391  p0=0.099 | p1=0.099  ETA=57s
   